# Go2 Course Homework: Public Colab Template

This notebook is a lightweight wrapper around the readable course code.

It assumes that the readable homework package is stored in a course GitHub repo.
Unlike the original payload-based notebook, this version keeps the important
source files visible as normal `.py` / `.json` files.

Recommended usage:
1. run setup
2. inspect the environment
3. show the key files
4. restore a checkpoint and render a demo
5. run the public benchmark

## 1. Configure repository URLs

Set `COURSE_REPO_URL` to the GitHub repository that contains this readable
homework package. The repo should contain:
- `train.py`
- `test_policy.py`
- `generate_public_rollout.py`
- `public_eval.py`
- `go2_pg_env/`
- `configs/course_config.json`

In [1]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/Shun-Hung/EEC289A_HW1.git"
COURSE_REPO_BRANCH = "main"
COURSE_REPO_DIR = Path("/content/go2_course_repo")

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = Path("/content/mujoco_playground")
UNITREE_DIR = Path("/content/unitree_mujoco")
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")

Running inside Colab.


## 2. Install system packages and clone repositories

In [2]:
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
import jax

# Tell Python to look inside the outer folder first
# 1. Force Python to forget the broken namespace package
if 'mujoco_playground' in sys.modules:
    del sys.modules['mujoco_playground']

# 2. Tell Python to look inside the outer folder first
if '/content/mujoco_playground' not in sys.path:
    sys.path.insert(0, '/content/mujoco_playground')

import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
if "/content/mujoco_playground/" not in mujoco_playground.__file__:
    raise RuntimeError("Expected mujoco_playground to be imported from /content/mujoco_playground")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/Shun-Hung/EEC289A_HW1.git /content/go2_course_repo
+ git clone https://github.com/deepmind/mujoco_menagerie.g

## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

In [3]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}

/content/go2_course_repo
Copied 16 assets into /content/go2_course_repo/go2_pg_env/xmls/assets


## 4. Inspect the environment before training

In [4]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2

/content/go2_course_repo
{
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "backend_impl": "jax",
  "control_dt": 0.02,
  "sim_dt": 0.004,
  "episode_length": 1000,
  "action_size": 12,
  "actor_obs_size": 48,
  "critic_obs_size": 123,
  "observation_layout": {
    "state": [
      [
        "local_linvel",
        3
      ],
      [
        "gyro",
        3
      ],
      [
        "gravity",
        3
      ],
      [
        "joint_pos_error",
        12
      ],
      [
        "joint_vel",
        12
      ],
      [
        "last_action",
        12
      ],
      [
        "command",
        3
      ]
    ],
    "privileged_state_extra": [
      [
        "gyro_clean",
        3
      ],
      [
        "accelerometer",
        3
      ],
      [
        "gravity_clean",
        3
      ],
      [
        "local_linvel_clean",
        3
      ],
      [
        "global_angvel",
        3
      ],
      [
        "joint_pos_error_clean",
        12
 

## 5. Read the most important files

In [ ]:
!sed -n '1,220p' go2_pg_env/joystick.py

In [ ]:
!sed -n '1,220p' train.py

In [ ]:
!sed -n '1,220p' benchmark_specs.py

## 6. Define a Colab-friendly runtime config

In [5]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000, #10_000_000
    "stage_2_num_timesteps": 5_000_000, #5_000_000
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)

wrote /content/go2_course_repo/configs/colab_runtime_config.json


In [14]:
!git config --global user.email "lshle@ucdavis.edu"
!git config --global user.name "Shun-Hung"

In [27]:
%cd /content/go2_course_repo
!git status
!git add .
!git commit -m "Describe your change"
!git push origin main

/content/go2_course_repo
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address


## 7. Dry-run the training config

In [6]:
!python train.py --config configs/colab_runtime_config.json --dry-run

{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "framework": "MuJoCo Playground + Brax PPO + MJX",
  "backend_impl": "jax",
  "actor_obs_key": "state",
  "critic_obs_key": "privileged_state",
  "use_domain_randomization": true,
  "seed": 0,
  "control": {
    "ctrl_dt": 0.02,
    "sim_dt": 0.004,
    "action_scale": 0.5,
    "action_type": "absolute_joint_position_target",
    "torque_mapping": "position_target_through_pd_actuator"
  },
  "course_budget": {
    "baseline_total_env_steps": 15000000,
    "leaderboard_max_env_steps": 30000000,
    "flat_terrain_only": true,
    "require_colab_gpu_runtime": true
  },
  "training_defaults": {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [
      256,
      256,
      128
    ],
    "value_hidden_layer_sizes": [
      256,
      256,
      128
    ]
  },
  

## 8. Run training

If you are demonstrating the pipeline interactively, it is usually better to
**skip this cell** and instead use a prepared checkpoint. Full training
includes JIT compile time.

In [23]:
!python train.py   --config configs/colab_runtime_config.json   --stage both  --output-dir artifacts/run_baseline

[run] output_dir=/content/go2_course_repo/artifacts/run_baseline
[run] stages=['stage_1', 'stage_2']
[stage_1] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=10000000 num_envs=1024 batch_size=256 num_evals=5
[stage_1] steps=0 eval_reward=0.005
[stage_1] steps=3276800 eval_reward=2.366
[stage_1] steps=6553600 eval_reward=21.870
[stage_1] steps=9830400 eval_reward=26.580
[stage_1] steps=13107200 eval_reward=29.014
[stage_1] finished: latest_checkpoint=/content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200 selected_checkpoint_source=/content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200
[stage_2] starting train: env=Go2JoystickFlatTerrain impl=jax target_steps=5000000 num_envs=1024 batch_size=256 num_evals=5
[stage_2] restoring from checkpoint: /content/go2_course_repo/artifacts/run_baseline/stage_1/checkpoints/000013107200
[stage_2] steps=0 eval_reward=26.723
[stage_2] steps=1638400 eval_reward=27.934
[stage_2] steps=3

## 9. Restore a checkpoint and render a deterministic demo

Replace the checkpoint path below with a real trained checkpoint if needed.

In [24]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_baseline" / "best_checkpoint"
DEMO_DIR = COURSE_REPO_DIR / "artifacts" / "demo_bundle"

!python test_policy.py  --config configs/colab_runtime_config.json  --checkpoint-dir {CHECKPOINT_DIR}   --stage-name stage_2    --output-dir {DEMO_DIR}

100% 284/284 [03:07<00:00,  1.52it/s]
{
  "homework_name": "Homework: Sim-to-Real Quadruped Locomotion with MuJoCo Playground",
  "robot": "Go2",
  "environment_name": "Go2JoystickFlatTerrain",
  "stage_name": "stage_2",
  "checkpoint_dir": "/content/go2_course_repo/artifacts/run_baseline/best_checkpoint",
  "num_steps": 283,
  "metrics": {
    "velocity_tracking_error": 0.1713978499174118,
    "yaw_tracking_error": 0.10100185871124268,
    "fall_rate": 1.0,
    "energy_proxy": 30.311561584472656,
    "foot_slip_proxy": 0.1013571247458458
  },
  "normalized_scores": {
    "velocity_tracking_error": 0.7960061430931092,
    "yaw_tracking_error": 0.9974953532218933,
    "fall_rate": 0.0,
    "energy_proxy": 0.3027637004852295,
    "foot_slip_proxy": 0.5480159736341901
  },
  "course_composite_score": 0.5783173731631703,
  "rollout_summary": {
    "video_path": "/content/go2_course_repo/artifacts/demo_bundle/demo.mp4",
    "rollout_npz": "/content/go2_course_repo/artifacts/demo_bundle/roll

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    '/content/go2_course_repo/artifacts/demo_bundle/demo.mp4',
    '/content/drive/MyDrive/demo.mp4'
)

## 10. Generate the public benchmark rollout

In [ ]:
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle"

# !python generate_public_rollout.py #   --config configs/colab_runtime_config.json #   --checkpoint-dir {CHECKPOINT_DIR} #   --stage-name stage_2 #   --output-dir {PUBLIC_DIR} #   --num-episodes 4 #   --render-first-episode

# !python public_eval.py #   --config configs/colab_runtime_config.json #   --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} #   --output-json {PUBLIC_DIR / "public_eval.json"}

## 11. Suggested walkthrough order

Recommended order:
1. `inspect_env.py`
2. `go2_pg_env/joystick.py`
3. `configs/course_config.json`
4. `test_policy.py`
5. `generate_public_rollout.py`
6. `public_eval.json`

This order mirrors the main workflow:
**task -> train -> restore -> benchmark**